## 1. Import reporting libraries and verify upstream evidence


In [1]:
import sys
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from common_metrics import require, schedule_metrics

DATA_DIR = Path("output")


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

from validation import verify_stage, validate_schedule
for upstream in range(5):
    verify_stage(upstream)


## 2. Load project evidence


In [2]:
def first_existing(paths):
    for path in paths:
        if Path(path).is_file():
            return Path(path)
    return None


paths = {
    "canonical": first_existing([DATA_DIR / "canonical_schedule.csv"]),
    "specification": first_existing([DATA_DIR / "model_specification.json"]),
    "optimized": first_existing([DATA_DIR / "optimized_schedule_fairness_aware.csv"]),
    "scenario_summary": first_existing([DATA_DIR / "optimization_scenario_summary.csv"]),
    "robustness": first_existing([DATA_DIR / "robustness_evidence.json"]),
}
missing = [k for k, v in paths.items() if v is None]
require(not missing, f"Missing required project evidence: {missing}. Run notebooks 00-04 first.")



source_hashes = {k: sha256_file(v) for k, v in paths.items()}
model_specification = json.loads(paths["specification"].read_text(encoding="utf-8"))
robustness_evidence = json.loads(paths["robustness"].read_text(encoding="utf-8"))
historical = pd.read_csv(paths["canonical"])
optimized = pd.read_csv(paths["optimized"])
scenario_summary = pd.read_csv(paths["scenario_summary"])

for frame in [historical, optimized]:
    frame["week_start"] = pd.to_datetime(frame["week_start"], errors="raise")
    frame["week_end"] = pd.to_datetime(frame["week_end"], errors="raise")
    for column in ["nT", "rotation_group", "rotation_round"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise").astype(int)

parameters = model_specification["parameters"]
derived_maps = model_specification["derived_maps"]
weekly_staffing_floor = int(parameters["weekly_staffing_floor"])
weekly_h24_required = int(parameters["weekly_h24_required"])
maximum_annual_workload = int(parameters["maximum_annual_workload"])
horizon_weeks = int(parameters["horizon_weeks"])
rotation_groups = [int(g) for g in parameters["rotation_groups"]]
group_sizes = {int(g): int(s) for g, s in derived_maps["group_sizes"].items()}
parameter_status = model_specification["parameter_status"]
structural_analysis = model_specification["structural_analysis"]

print("Loaded evidence chain from notebooks 00-04.")
print("Historical / optimized assignments:", len(historical), len(optimized))


Loaded evidence chain from notebooks 00-04.
Historical / optimized assignments: 931 923


## 3. Internal consistency check


In [3]:
pharmacies = sorted(derived_maps["primary_group_by_pharmacy"])
require(source_hashes["canonical"] == model_specification["canonical_sha256"], "Canonical lineage mismatch.")
for key, evidence_key in [("canonical","canonical_sha256"),("specification","model_specification_sha256"),("optimized","optimized_schedule_sha256")]:
    require(source_hashes[key] == robustness_evidence[evidence_key], f"Robustness {key} lineage mismatch.")
require(json.loads((DATA_DIR / "applied_model_specification.json").read_text()) == model_specification, "Applied specification mismatch.")
validate_schedule(optimized, historical, model_specification)
historical_metrics = schedule_metrics(historical,historical[["week_id","nT"]].drop_duplicates(),pharmacies)
optimized_metrics = schedule_metrics(optimized,optimized[["week_id","nT"]].drop_duplicates(),pharmacies)
selected_scenario_name = "Fairness-aware"
metric_columns = {"assignments":"assignments","total_min":"total_minimum","total_max":"total_maximum",
                  "total_range":"total_range","total_jain":"total_jain_index","total_gini":"total_gini",
                  "h24_min":"h24_minimum","h24_max":"h24_maximum","h24_range":"h24_range",
                  "h24_jain":"h24_jain_index","h24_gini":"h24_gini"}
scenario_files = {"Historical":paths["canonical"],"Efficiency-first":DATA_DIR/"optimized_schedule_efficiency_first.csv",
                  "Fairness-aware":paths["optimized"],"Full-group":DATA_DIR/"optimized_schedule_full_group.csv"}
require(set(scenario_summary.scenario) == set(scenario_files) and scenario_summary.scenario.is_unique, "Scenario names mismatch.")
for name, path in scenario_files.items():
    frame=pd.read_csv(path)
    if name != "Historical":validate_schedule(frame,historical,model_specification)
    metrics=schedule_metrics(frame,frame[["week_id","nT"]].drop_duplicates(),pharmacies)
    row=scenario_summary.set_index("scenario").loc[name]
    for column,key in metric_columns.items():
        require(np.isclose(row[column],metrics[key],rtol=1e-10,atol=1e-12),f"Scenario {name}: {column} mismatch.")
    require(row.reduction_vs_history == len(historical)-len(frame), "Assignment reduction mismatch.")
consistency_checks = pd.DataFrame([
    ("All scenario metrics match materialized schedules",True,"4 scenarios; all exported metrics"),
    ("Weekly staffing floor satisfied",True,f"minimum weekly active = {optimized.groupby('week_id').size().min()}"),
    ("All modeled schedule constraints",True,"metadata, unique assignments, eligibility, staffing, H24, cycle bounds, workload, repeated patterns"),
    ("Evidence lineage",True,"stage manifests and cross-artifact hashes checked"),
],columns=["check","passed","evidence"])
consistency_checks["status"]="PASS"
display(consistency_checks[["check","status","evidence"]])


,check,status,evidence
0,All scenario metrics match materialized schedules,PASS,4 scenarios; all exported metrics
1,Weekly staffing floor satisfied,PASS,minimum weekly active = 17
2,All modeled schedule constraints,PASS,"metadata, unique assignments, eligibility, sta..."
3,Evidence lineage,PASS,stage manifests and cross-artifact hashes checked


## 4. Explain pattern and pharmacy changes


In [4]:

nt_values=sorted(historical.nT.unique())
change_rows=[]
for nt in nt_values:
    a=historical[historical.nT==nt];b=optimized[optimized.nT==nt]
    active_a,active_b=set(a.pharmacy_id),set(b.pharmacy_id)
    h24_a=set(a.loc[a.shift_type.eq("H24"),"pharmacy_id"])
    h24_b=set(b.loc[b.shift_type.eq("H24"),"pharmacy_id"])
    cols=["pharmacy_id","location_id","shift_type","access_mode"]
    changed=set(a[cols].itertuples(index=False,name=None)) != set(b[cols].itertuples(index=False,name=None))
    if changed:
        change_rows.append({"nT":int(nt),"historical_count":len(active_a),"optimized_count":len(active_b),
            "active_set_changed":active_a!=active_b,"h24_set_changed":h24_a!=h24_b,
            "added":json.dumps(sorted(active_b-active_a)),"removed":json.dumps(sorted(active_a-active_b)),
            "h24_added":json.dumps(sorted(h24_b-h24_a)),"h24_removed":json.dumps(sorted(h24_a-h24_b))})
changed_nt=pd.DataFrame(change_rows,columns=["nT","historical_count","optimized_count","active_set_changed","h24_set_changed","added","removed","h24_added","h24_removed"])
display(changed_nt)
print(f"Changed full patterns: {len(changed_nt)} / {len(nt_values)}; active sets: {changed_nt.active_set_changed.sum()}; H24 sets: {changed_nt.h24_set_changed.sum()}")


Changed full patterns: 11 / 42; active sets: 9; H24 sets: 3


,nT,historical_count,optimized_count,active_set_changed,h24_set_changed,added,removed,h24_added,h24_removed
0,9,18,17,True,False,[],"[""P0021""]",[],[]
1,10,18,17,True,False,[],"[""P0049""]",[],[]
2,16,18,17,True,False,[],"[""P0020""]",[],[]
3,17,18,17,True,False,[],"[""P0045""]",[],[]
4,21,17,17,True,True,"[""P0111""]","[""P0076""]","[""P0111""]","[""P0076""]"
5,22,17,17,False,True,[],[],"[""P0009""]","[""P0003""]"
6,23,18,17,True,False,[],"[""P0029""]",[],[]
7,24,18,17,True,False,[],"[""P0039""]",[],[]
8,25,17,17,False,True,[],[],"[""P0057""]","[""P0067""]"
9,30,18,17,True,False,[],"[""P0034""]",[],[]


## 5. Compare scenarios and select the recommendation


In [5]:
comparison_columns = ["assignments", "total_range", "h24_range"]


def is_dominated(row, frame):
    '''A scenario is Pareto-dominated if some other scenario is at least as good on every
    metric and strictly better on at least one.'''
    others = frame.drop(index=row.name)
    weakly_better = others[comparison_columns].le(row[comparison_columns].to_numpy()).all(axis=1)
    strictly_better = others[comparison_columns].lt(row[comparison_columns].to_numpy()).any(axis=1)
    return bool((weakly_better & strictly_better).any())


scenario_comparison = scenario_summary.copy()
scenario_comparison["pareto_efficient"] = ~scenario_comparison.apply(lambda r: is_dominated(r, scenario_comparison), axis=1)
scenario_comparison = scenario_comparison.sort_values(
    model_specification["recommendation_order"] + ["scenario"]
).reset_index(drop=True)
recommended_scenario = str(scenario_comparison.iloc[0]["scenario"])
require(recommended_scenario == selected_scenario_name, "Exported schedule does not match the documented decision rule.")
scenario_comparison["recommended"] = scenario_comparison["scenario"].eq(recommended_scenario)
display(scenario_comparison[["scenario", "assignments", "total_range", "h24_range", "pareto_efficient", "recommended"]])


,scenario,assignments,total_range,h24_range,pareto_efficient,recommended
0,Fairness-aware,923,1,1,True,True
1,Full-group,931,1,1,False,False
2,Historical,931,2,3,False,False
3,Efficiency-first,901,3,1,True,False


## 6. Final validation, robustness interpretation, and report


In [6]:
robustness_results=robustness_evidence["results"]
robustness_group_summary=pd.DataFrame(robustness_evidence["group_summary"])
plan=pd.read_csv(DATA_DIR/"single_absence_response_plan.csv",keep_default_na=False)
scheduled=plan[plan.was_scheduled_active]
require(len(plan)==robustness_results["tested_pattern_absences"],"Response count mismatch.")
require(len(scheduled)==robustness_results["scheduled_active_absences"],"Scheduled count mismatch.")
require(int(scheduled.hard_constraints_recoverable_within_group.sum())==robustness_results["recoverable_within_group"],"Recovery count mismatch.")
require(int(scheduled.cross_group_authorization_required.sum())==robustness_results["total_requiring_human_intervention"],"Manual count mismatch.")
require(int(scheduled.secondary_eligible_candidate_from_history.ne("").sum())==robustness_results["historical_candidates_requiring_approval"],"Candidate count mismatch.")
require(len(pd.read_csv(DATA_DIR/"authorized_substitutions_template.csv"))==robustness_results["total_requiring_human_intervention"],"Authorization template omits events.")
final_validation_checklist=pd.DataFrame([
    ("Data and implementation lineage","Stage manifests and explicit canonical/specification/optimized/robustness links verified","PASS"),
    ("All scenario schedules","All exported schedule constraints and summary metrics checked","PASS"),
    ("Pattern-level recovery","Every reported same-group success and historical candidate materialized and checked","PASS"),
    ("Cross-group boundary",f"{robustness_results['total_requiring_human_intervention']} simulated cases need a cross-group decision; no approval records are invented","DOCUMENTED MODEL OUTCOME"),
    ("Group bound",structural_analysis["conclusion"],"PROVEN UNDER STATED ASSUMPTIONS"),
    ("Staffing and workload thresholds","Derived from source minima/maxima and retained by explicit continuity policy","JUSTIFIED MODEL CHOICE"),
    ("Calendar boundary","Final block ends 2027-01-03, six elapsed days after Monday start; source preserved and limitation documented","CAVEAT"),
    ("Dataset-only scope","No geography, actual availability, demand, preferences or external rules added; excluded by project scope","SCOPE SATISFIED"),
    ("Recovery scope","Pattern-level, one absence at a time; H24 moves may affect other patterns, total fairness is not locked","CAVEAT"),
],columns=["project_component","evidence","status"])
display(final_validation_checklist)
from reporting import build_report
final_report=build_report(historical_metrics,optimized_metrics,scenario_comparison,changed_nt,
                         robustness_evidence,model_specification,
                         json.loads((DATA_DIR/"observed_data_profile.json").read_text()),
                         final_validation_checklist)
print(final_report)


# Fair and Constraint-Aware Pharmacy Shift Scheduling

## Abstract

This project investigates pharmacy duty scheduling as a constrained allocation of workload. The sole empirical input is the supplied TURNI CSV. The Bologna city-area calendar contains 53 weekly blocks, 123 rotating pharmacies and 931 assignments, organized into 42 recurring patterns and seven dominant rotation groups. A mixed-integer linear program preserves a historical staffing floor, H24 duty count, primary-group eligibility, a horizon workload ceiling and a policy of balanced H24 cycle counts. It compares efficiency-first, fairness-aware and full-group scenarios using lexicographic priorities.

The selected fairness-aware schedule has 923 assignments, total workload range 1 and H24 range 1, compared with 931, 2 and 3 historically. Total Jain improves slightly, but total Gini increases slightly; fairness improvement is metric-specific. 11 complete nT patterns change, including 2 with H24-only changes.

The robustnes

,project_component,evidence,status
0,Data and implementation lineage,Stage manifests and explicit canonical/specifi...,PASS
1,All scenario schedules,All exported schedule constraints and summary ...,PASS
2,Pattern-level recovery,Every reported same-group success and historic...,PASS
3,Cross-group boundary,306 simulated cases need a cross-group decisio...,DOCUMENTED MODEL OUTCOME
4,Group bound,PROVEN: no repartition of 123 pharmacies into ...,PROVEN UNDER STATED ASSUMPTIONS
5,Staffing and workload thresholds,Derived from source minima/maxima and retained...,JUSTIFIED MODEL CHOICE
6,Calendar boundary,"Final block ends 2027-01-03, six elapsed days ...",CAVEAT
7,Dataset-only scope,"No geography, actual availability, demand, pre...",SCOPE SATISFIED
8,Recovery scope,"Pattern-level, one absence at a time; H24 move...",CAVEAT


## 7. Export the final project artifacts


In [7]:


(DATA_DIR / "final_project_report.md").write_text(final_report, encoding="utf-8")
changed_nt.to_csv(DATA_DIR / "nt_change_explanation.csv", index=False)
scenario_comparison.to_csv(DATA_DIR / "scenario_pareto_comparison.csv", index=False)
final_validation_checklist.to_csv(DATA_DIR / "final_validation_checklist.csv", index=False)

final_project_evidence = {
    "evidence_type": "final_project_comparison",
    "source_hashes": source_hashes,
    "recommended_scenario": recommended_scenario,
    "historical_metrics": historical_metrics,
    "optimized_metrics": optimized_metrics,
    "robustness_results": robustness_results,
    "structural_analysis": structural_analysis,
    "geography_used": False,
}
Path(DATA_DIR / "final_project_evidence.json").write_text(
    json.dumps(final_project_evidence, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Saved: output/final_project_report.md and supporting CSV/JSON evidence files.")

from validation import write_stage
write_stage(5, ['output/stage_00_manifest.json', 'output/stage_01_manifest.json', 'output/stage_02_manifest.json', 'output/stage_03_manifest.json', 'output/stage_04_manifest.json'], ['output/final_project_report.md', 'output/nt_change_explanation.csv', 'output/scenario_pareto_comparison.csv', 'output/final_validation_checklist.csv', 'output/final_project_evidence.json'])


Saved: output/final_project_report.md and supporting CSV/JSON evidence files.
